# Temporal-Difference Methods — FrozenLake

Temporal-difference (TD) methods for control without a model of $T$ and $R$: online learning that combines Monte Carlo (learn from experience) and dynamic programming (bootstrap).

## Goal
Learn $q_\pi$ / $q_\star$ and improve policies **without** knowing $T$ and $R$, updating after each transition (bootstrapping), without waiting for the end of the episode.

## Environment
Gymnasium `FrozenLake-v1` with `is_slippery=True`.  
$4\times 4$ grid: start `S`, frozen `F`, holes `H`, goal `G`.  
Actions: `0` = Left, `1` = Down, `2` = Right, `3` = Up.  
Reward $+1$ only when reaching `G`, else $0$. Slippery ice makes transitions stochastic.

## Implemented below
1. Helpers: evaluate, play, $\varepsilon$-greedy
2. SARSA (on-policy TD control)
3. Q-learning (off-policy TD control)
4. Comparison SARSA vs Q-learning


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import time

from IPython.display import clear_output, display

np.random.seed(42)
env = gym.make("FrozenLake-v1", is_slippery=True)


## Reminder: TD between DP and Monte Carlo

TD methods:

- learn directly from **experience** (like MC) — no model of $T$ or $R$;
- can learn **online** on incomplete episodes (like DP);
- update an estimate from other estimates: **bootstrap** (like DP).

**Policy evaluation TD(0).** From Bellman,
$$v_\pi(s) = \mathbb{E}_\pi\bigl[R_{t+1} + \gamma v_\pi(S_{t+1}) \bigm| S_t=s\bigr],$$
Monte Carlo uses the full return $G_t$; TD(0) replaces it by the bootstrapped target:
$$v(S_t) \leftarrow v(S_t) + \alpha\bigl[R_{t+1} + \gamma v(S_{t+1}) - v(S_t)\bigr].$$
The quantity $\delta_t = R_{t+1} + \gamma v(S_{t+1}) - v(S_t)$ is the **temporal-difference error**.

For **control**, we learn $q$ instead of $v$. Below: SARSA (on-policy) and Q-learning (off-policy).


## Helpers

**`evaluate`.** Success rate of the greedy policy w.r.t. $Q$ over many episodes (reward $=1$ only if the goal is reached).

**`play`.** Visualize a few greedy episodes with `render_mode="rgb_array"`.

**`epsilon_greedy`.** With probability $\varepsilon$, pick a random action; otherwise $\arg\max_a Q(s,a)$.


In [ ]:
def evaluate(env, Q, num_episodes=1000):
    wins = 0
    for _ in range(num_episodes):
        s, _ = env.reset()
        done = False
        while not done:
            a = int(np.argmax(Q[s]))  # greedy
            s, r, terminated, truncated, _ = env.step(a)
            done = terminated or truncated
            wins += r  # r == 1 only if the goal is reached
    return wins / num_episodes


def play(Q, n_episodes=3, delay=0.3):
    viz = gym.make("FrozenLake-v1", is_slippery=True, render_mode="rgb_array")
    for ep in range(n_episodes):
        s, _ = viz.reset()
        done = False
        total_r = 0
        while not done:
            frame = viz.render()
            clear_output(wait=True)
            plt.figure(figsize=(4, 4))
            plt.imshow(frame)
            plt.title(f"épisode {ep+1}  |  reward={total_r}")
            plt.axis("off")
            display(plt.gcf())
            plt.close()
            a = int(np.argmax(Q[s]))
            s, r, terminated, truncated, _ = viz.step(a)
            done = terminated or truncated
            total_r += r
            time.sleep(delay)
        clear_output(wait=True)
        plt.figure(figsize=(4, 4))
        plt.imshow(viz.render())
        plt.title(f"épisode {ep+1} terminé  |  reward={total_r}")
        plt.axis("off")
        display(plt.gcf())
        plt.close()
        time.sleep(0.8)
    viz.close()


def epsilon_greedy(env, Q, s, epsilon):
    if np.random.rand() < epsilon:
        return env.action_space.sample()
    return int(np.argmax(Q[s]))


## SARSA (on-policy TD control)

Learn $q$ while following the **same** soft policy that generates the data (typically $\varepsilon$-greedy).

Update uses the action $A'$ **actually chosen** next:
$$q(S,A) \leftarrow q(S,A) + \alpha\bigl[R + \gamma q(S',A') - q(S,A)\bigr].$$
If $S'$ is terminal, the target is just $R$ (no bootstrap).

Name **SARSA** = State–Action–Reward–State–Action: each update needs the quintuple $(S,A,R,S',A')$.

Under GLEI (*Glouton à la Limite avec Exploration Infinie*) and suitable step sizes $\alpha_t$, SARSA converges to $q_\star$.


In [ ]:
def Sarsa(env, num_episodes, alpha, gamma=1.0, epsilon=0.1):
    Q = np.zeros((env.observation_space.n, env.action_space.n))

    for _ in range(num_episodes):
        s = env.reset()[0]
        a = epsilon_greedy(env, Q, s, epsilon)

        while True:
            s_prime, r, terminated, truncated, _ = env.step(a)

            if terminated or truncated:
                Q[s, a] += alpha * (r - Q[s, a])
                break

            a_prime = epsilon_greedy(env, Q, s_prime, epsilon)
            Q[s, a] += alpha * (r + gamma * Q[s_prime, a_prime] - Q[s, a])
            s = s_prime
            a = a_prime

    return Q


## Train SARSA

Train on FrozenLake, print $Q$, success rate of the greedy policy, then play a few episodes.


In [ ]:
Q = Sarsa(env, num_episodes=20_000, alpha=0.1, gamma=0.99, epsilon=0.1)
print(Q)
print("success rate:", evaluate(env, Q))


In [ ]:
play(Q, n_episodes=5, delay=0.25)


## Q-learning (off-policy TD control)

Learn $q_\star$ largely **independently** of the behavior policy.

The target uses a **greedy** (max) backup, while actions may still be chosen by an exploratory behavior policy ($\varepsilon$-greedy):
$$q(S,A) \leftarrow q(S,A) + \alpha\Bigl[R + \gamma \max_{a'} q(S',a') - q(S,A)\Bigr].$$

Difference with SARSA: SARSA bootstraps with $q(S',A')$ (on-policy); Q-learning bootstraps with $\max_{a'} q(S',a')$ (off-policy target).

Under GLEI and the usual conditions on $(\alpha_t)$, Q-learning converges to $q_\star$.


In [ ]:
def Q_learning(env, num_episodes, alpha, gamma=1.0, epsilon=0.1):
    Q = np.zeros((env.observation_space.n, env.action_space.n))

    for _ in range(num_episodes):
        s = env.reset()[0]

        while True:
            a = epsilon_greedy(env, Q, s, epsilon)
            s_prime, r, terminated, truncated, _ = env.step(a)

            if terminated or truncated:
                Q[s, a] += alpha * (r - Q[s, a])
                break

            Q[s, a] += alpha * (r + gamma * np.max(Q[s_prime]) - Q[s, a])
            s = s_prime

    return Q


## Train Q-learning

Same hyperparameters as SARSA for a fair comparison. Success rates can differ run-to-run because of stochastic ice and exploration.


In [ ]:
Q_prime = Q_learning(env, num_episodes=20_000, alpha=0.1, gamma=0.99, epsilon=0.1)
print(Q_prime)
print("success rate:", evaluate(env, Q_prime))


In [ ]:
play(Q_prime, n_episodes=5, delay=0.25)


## SARSA vs Q-learning

Under GLEI, both converge to an optimal solution, but learning trajectories differ.

Classic **cliff walking** intuition (course): reward $-1$ per step, $-100$ for falling. SARSA (on-policy) tends to learn a **safer** path first (away from the cliff), while Q-learning estimates $q_\star$ more directly and prefers the optimal path along the edge — at the cost of more falls during learning if behavior stays exploratory.

On FrozenLake, both should reach a strong greedy policy; compare success rates above (results vary with seed and $\varepsilon$).

### Exploration vs exploitation

Collected data **depends** on the agent's behavior. Common strategies: $\varepsilon$-greedy (used here), softmax / Boltzmann, optimistic initialization. Guarantees often require $\varepsilon$ to decay appropriately (GLEI).


## References

1. S. Airiau. *Apprentissage par renforcement*. M2 IASD, Université Paris Dauphine. Lecture 5: *Méthodes de différences temporelles*. [https://www.lamsade.dauphine.fr/~airiau/Teaching/M2-IASDapp-RL/](https://www.lamsade.dauphine.fr/~airiau/Teaching/M2-IASDapp-RL/)
2. C. J. C. H. Watkins. *Learning from Delayed Rewards*. PhD thesis, University of Cambridge, 1989.
3. R. S. Sutton and A. G. Barto. *Reinforcement Learning: An Introduction*, 2nd ed. MIT Press, 2018.
